# Data Preperation for News Classification

In [2]:
import collections
import numpy as np
import pandas as pd
import re

from argparse import Namespace

In [3]:
args = Namespace(
    raw_dataset_csv="data/ag_news/news.csv",
    train_proportion=0.7,
    val_proportion=0.15,
    test_proportion=0.15,
    output_munged_csv="data/ag_news/news_with_splits.csv",
    seed=1337
)

In [4]:
# Read raw data
news = pd.read_csv(args.raw_dataset_csv, header=0)

In [5]:
news.head()

,category,title
0,Business,Wall St. Bears Claw Back Into the Black (Reuters)
1,Business,Carlyle Looks Toward Commercial Aerospace (Reu...
2,Business,Oil and Economy Cloud Stocks' Outlook (Reuters)
3,Business,Iraq Halts Oil Exports from Main Southern Pipe...
4,Business,"Oil prices soar to all-time record, posing new..."


In [6]:
# Unique classes
set(news.category)

{'Business', 'Sci/Tech', 'Sports', 'World'}

In [7]:
# Splitting train by category
# Create dict
by_category = collections.defaultdict(list)
for _, row in news.iterrows():
    by_category[row.category].append(row.to_dict())

In [8]:
by_category 

defaultdict(list,
            {'Business': [{'category': 'Business',
               'title': 'Wall St. Bears Claw Back Into the Black (Reuters)'},
              {'category': 'Business',
               'title': 'Carlyle Looks Toward Commercial Aerospace (Reuters)'},
              {'category': 'Business',
               'title': "Oil and Economy Cloud Stocks' Outlook (Reuters)"},
              {'category': 'Business',
               'title': 'Iraq Halts Oil Exports from Main Southern Pipeline (Reuters)'},
              {'category': 'Business',
               'title': 'Oil prices soar to all-time record, posing new menace to US economy (AFP)'},
              {'category': 'Business',
               'title': 'Stocks End Up, But Near Year Lows (Reuters)'},
              {'category': 'Business',
               'title': 'Money Funds Fell in Latest Week (AP)'},
              {'category': 'Business',
               'title': 'Fed minutes show dissent over inflation (USATODAY.com)'},
             

In [9]:
# Create split data
final_list = []
np.random.seed(args.seed)
for _, item_list in sorted(by_category.items()):
    np.random.shuffle(item_list)
    n = len(item_list)
    n_train = int(args.train_proportion*n)
    n_val = int(args.val_proportion*n)
    n_test = int(args.test_proportion*n)
    
    # Give data point a split attribute
    for item in item_list[:n_train]:
        item['split'] = 'train'
    for item in item_list[n_train:n_train+n_val]:
        item['split'] = 'val'
    for item in item_list[n_train+n_val:]:
        item['split'] = 'test'  
    
    # Add to final list
    final_list.extend(item_list)

In [10]:
# Write split data to file
final_news = pd.DataFrame(final_list)

In [11]:
final_news.split.value_counts()

split
train    84000
val      18000
test     18000
Name: count, dtype: int64

In [12]:
# Preprocess the reviews
def preprocess_text(text):
    text = ' '.join(word.lower() for word in text.split(" "))
    text = re.sub(r"([.,!?])", r" \1 ", text)
    text = re.sub(r"[^a-zA-Z.,!?]+", r" ", text)
    return text
    
final_news.title = final_news.title.apply(preprocess_text)

In [13]:
final_news.head()

,category,title,split
0,Business,"jobs , tax cuts key issues for bush",train
1,Business,jarden buying mr . coffee s maker,train
2,Business,retail sales show festive fervour,train
3,Business,intervoice s customers come calling,train
4,Business,boeing expects air force contract,train


In [14]:
# Write munged data to CSV
final_news.to_csv(args.output_munged_csv, index=False)